# Using tissue models in Arritmic3D

In this notebook we will show how we can use different restitution models in different regions of the tissue.

We start by importing the arritmic3d module and other required packages.

In [ ]:
%pip install arritmic3d

In [ ]:
import os
import time
import shutil
import pyvista as pv
import matplotlib.pyplot as plt
from google.colab import files
import arritmic3d as a3d

# Flag to download the simulated cases.
# Set to True if you want the results to
# be downloaded, for inspection with paraview
download_cases = True

We define basic plotting functions that we will use to visualize the results.

In [ ]:
def plot_grid(grid,field="AP",plt_show=False,title = "") :
    rng = ranges.get(field,None)
    # Apply a threshold to the grid. Cells with restitution_model==0
    # are not part of the simulation domain
    grid = grid.threshold(0.5, scalars="restitution_model", all_scalars=True)
    # Setup plotter for a static image
    plotter = pv.Plotter(off_screen=True)
    plotter.add_mesh(grid, scalars=field, cmap="coolwarm", show_edges=True,rng=rng)
    plotter.view_xy()
    img = plotter.show(screenshot=True)
    # Display using matplotlib
    plt.imshow(img)
    plt.axis('off')
    if title != "":
        plt.title(title)
    if plt_show:
        plt.show()
    return img

def plot_vtk(file_path, field="AP",plt_show=False,title = ""):
    grid = pv.read(file_path)
    plot_grid(grid,field=field,plt_show=plt_show,title=title)

def plot_animation(case_dir, field="AP", init_time=None, end_time=None, step=None):
    config = a3d.load_case_config(case_dir)
    if init_time is None:
        init_time = config.get("VTK_OUTPUT_INITIAL_TIME")
    if step is None:
        step = config.get("VTK_OUTPUT_PERIOD")
    if end_time is None:
        end_time = config.get("SIMULATION_DURATION")

    for time_ms in range(init_time, end_time + 1, step):
        file_path = f"{case_dir}/slab_{time_ms:05d}.vtu"
        if os.path.exists(file_path):
            plot_vtk(file_path, field=field, plt_show=True, title=f"Time: {time_ms} ms")
            # Small pause to allow the UI to update
            time.sleep(0.01)

ranges = {
    "AP": [-80, 40],
    "APD": [100,400],
    "CV": [0.1,0.8],
    "State": [0, 2]
}

def download_case(case_path):
    print(" Packaging results...")
    if os.path.exists(case_path) and len(os.listdir(case_path)) > 0:
        zip_name = "/content/case.zip"
        !zip -q -FSr {zip_name} {case_path}
        print(" Downloading results...")
        files.download(zip_name)
        print(f" DOWNLOAD STARTED: {zip_name}")
    else:
        print(" Error: Simulation produced no files.")

## Build a slab

In the previous example we let arritmic3d build a test case. Now we are going to build the case controlling its properties.

We start by building a slab similar to the one in the first tutorial: a square slab of healthy tissue (`restitution_model=2`) and a central square region with border zone (BZ) tissue (`restitution_model=5`).

The function we use to build the slab is (surprise...) `build_slab()`. It generates the tissue slab and saves it to a `vtk` file.

This function receives the command line arguments to configure and save the slab. In short, we build a list of strings with the path to the output file followed by a series of command line options. In our example we are going to define the slab geometry, set a base restitution model and add a central region with a different restitution model.

For a detailed description of the arguments that can be used, please, read the [documentation](https://commlabuv.github.io/arritmic3d/#generating-tissue-slabs).


In [ ]:
# We start by defining the output directory
output_dir="test_slab"

# Remove the directory if it exists (for notebook repeatability)
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Ensure the subfolder for the slab exists and set the output file name
os.makedirs(os.path.join(output_dir, "input_data"), exist_ok=True)
slab_path = os.path.join(output_dir, "input_data", "slab.vtk")

# Define the list of arguments.
# Check https://commlabuv.github.io/arritmic3d/#generating-tissue-slabs
slab_args = [
    slab_path,
    "--nnodes", "70", "70", "2",
    "--spacing", "0.1", "0.1", "0.1",
    "--region-by-side", "south", "1",
    "--field", "restitution_model", "2",
    "--region", '{"shape" : "square", "cx" : 3.5, "cy" : 3.5, "r1" : 2.0, "r2" : 2.0, "restitution_model" : 5}'
]

# Now, we can build the slab with these arguments
a3d.build_slab(args_list=slab_args)

# We plot the slab
plot_vtk(slab_path,field="restitution_model")



## Configure the simulation

Now, we set the parameters of the simulation. The parameters are set in a dictionary with the same structure as the json configuration file. We do not need to set all the parameters. Any parameter that is not defined in the dictionary will be set to its default value.
Read the [documentation](https://commlabuv.github.io/arritmic3d/#sec-config-file) for a detailed list of configuration parameters.

We are going to simulate during 1000ms, with an initial stimulus at 0 ms and a pacing protocol of a fixed BCL of 400ms. We are going to save the state every 10 ms.

In [ ]:
# Configure the simulation
print("\n--- Configuring simulation ---")
config = {
    "VTK_INPUT_FILE": slab_path,
    "APD_MODEL": "TenTuscher",
    "CV_MODEL": "TenTuscher",
    "SIMULATION_DURATION": 1000,
    "VTK_OUTPUT_PERIOD": 10,
    "PROTOCOL": [
        {
            "ACTIVATION_REGION": 1,
            "FIRST_ACTIVATION_TIME": 0,
            "N_STIMS_PACING": [3],
            "BCL": [400]
        }
    ]
}

## Run the simulation

We run the simulation on the output directory where we have saved the slab, using the configuration dictionary. We visualize the state of the slab after 40ms and after 80ms.

We will see that the propagation of the activation is slower in the central square, that corresponds to BZ.

In [ ]:
# Run the simulation
a3d.arritmic3d(output_dir,config)

# Download the simulation if requested
if download_cases:
    download_case(output_dir)

In [ ]:
# Visualize the slab at t=10ms and t = 20ms
# We use plt_show = True to ensure visualization of both plots
plot_vtk(os.path.join(output_dir,"slab_00010.vtu"), plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_00020.vtu"))


## Tissue stabilization

If we maintain the BCL, the state of the tissue should, eventually, reach a stationary state that does not change from beat to beat. However, the tissue is not initialized to that state at the beginning of the simulation.

Since the activation depends on previous APD and DI values, it takes a few cycles to reach the stable state of the slab. We can check this by inspecting the state of the slab 40ms after each activation: at t equal to 40, 440 and 840 ms.

We will visualize different variables of the tissue state.

In [ ]:
# Visualize the AP at t = 10, 410 and 810
plot_vtk(os.path.join(output_dir,"slab_00010.vtu"), plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_00410.vtu"), plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_00810.vtu"))


In [ ]:
# Visualize APD and CV at t = 10, 410 and 810
plot_vtk(os.path.join(output_dir,"slab_00010.vtu"),field="APD", plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_00410.vtu"),field="APD", plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_00810.vtu"),field="APD", plt_show = True)

plot_vtk(os.path.join(output_dir,"slab_00010.vtu"),field="CV", plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_00410.vtu"),field="CV", plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_00810.vtu"),field="CV")


### Stabilizing the tissue

We can let the tissue stabilize by simulating more beats. We extend the simulation duration and the number of stimuli in the protocol. It usually takes between 12 and 20 beats at constant BCL to stabilize, depending on tissue size and properties.

In [ ]:
# Reconfigure the simulation
config["SIMULATION_DURATION"] = 5000
config["PROTOCOL"][0]["N_STIMS_PACING"] = 13

# Run the simulation
a3d.arritmic3d(output_dir,config)

# Download the simulation if requested
if download_cases:
    download_case(output_dir)

Now, if we plot the APD and the CV of the tissue in two consecutive beats we see how the values have stabilized.

In [ ]:
# Visualize the APD and DI at t = 4420 and 4820
plot_vtk(os.path.join(output_dir,"slab_04420.vtu"), field="APD", plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_04820.vtu"), field="APD", plt_show = True)

plot_vtk(os.path.join(output_dir,"slab_04420.vtu"), field="CV", plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_04820.vtu"), field="CV")



## Seting a core region

We conclude this tutorial by defining a core tissue region. We can do this by setting the `restitution_model` field to 0 for some cells. This will exclude that tissue from the tissue domain and, thus, prevent it from activating.

We make the change in the `slab_args` list of arguments, where we add a new `--region` option. The result is a new slab, with a 'hole' in its center.

In [ ]:
# We define the output directory
output_dir="test_slab_core"

# Remove the directory if it exists (for notebook repeatability)
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Ensure the subfolder for the slab exists and set the output file name
os.makedirs(os.path.join(output_dir, "input_data"), exist_ok=True)
slab_path = os.path.join(output_dir, "input_data", "slab.vtk")

# Define the list of arguments.
# Check https://commlabuv.github.io/arritmic3d/#generating-tissue-slabs
slab_args = [
    slab_path,
    "--nnodes", "45", "45", "5",
    "--spacing", "0.1", "0.1", "0.1",
    "--region-by-side", "south", "1",
    "--field", "restitution_model", "2",
    "--region", '{"shape" : "square", "cx" : 2.25, "cy" : 2.25, "r1" : 1.0, "r2" : 1.0, "restitution_model" : 5}',
    "--region", '{"shape" : "square", "cx" : 2.25, "cy" : 2.25, "r1" : 0.5, "r2" : 0.5, "restitution_model" : 0}'
]

# Now, we can build the slab with these arguments
a3d.build_slab(args_list=slab_args)

# We plot the slab
plot_vtk(slab_path,field="restitution_model")



We wil stabilize the tissue and, then, show the results. We start saving the output after ending the stabilization, and save with higher temporal resolution (every 2ms). We apply 11 activations with BCL=400 (at times, 0,400,...,4000) and save the depolarization of last beat, from 4000ms to 4050ms. Storing only the final part of the simulation saves a lot of space and computation time.

In [ ]:
# Configure the simulation
print("\n--- Configuring simulation ---")
config = {
    "VTK_INPUT_FILE": slab_path,
    "APD_MODEL": "TenTuscher",
    "CV_MODEL": "TenTuscher",
    "SIMULATION_DURATION": 4050,
    "VTK_OUTPUT_PERIOD": 2,
    "VTK_OUTPUT_INITIAL_TIME": 4000,
    "PROTOCOL": [
        {
            "ACTIVATION_REGION": 1,
            "FIRST_ACTIVATION_TIME": 0,
            "N_STIMS_PACING": [11],
            "BCL": [400]
        }
    ]
}

# Run the simulation
a3d.arritmic3d(output_dir,config)

# Download the simulation if requested
if download_cases:
    download_case(output_dir)

Finally, we show some snapshots of the simulation.

In this case, we see how the BZ region still slows down the propagation of the activation, and how the core zone does not propagate.


In [ ]:
# Visualize the AP at t = 4002, 4004 and 4006
plot_vtk(os.path.join(output_dir,"slab_04002.vtu"), plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_04004.vtu"), plt_show = True)
plot_vtk(os.path.join(output_dir,"slab_04006.vtu"))
